In [11]:
from sklearn.neighbors import BallTree
import pandas as pd
import numpy as np

ab_filtered = pd.read_csv("AB_NYC_2019_cleaned.csv")
mta_distance = pd.read_csv('MTA_Subway_Stations_distance.csv')



In [ ]:

ab_final = ab_filtered.copy()
mta_final = mta_distance.copy()


airbnb_coords = np.radians(ab_final[["latitude", "longitude"]].values)
subway_coords = np.radians(mta_final[["GTFS Latitude", "GTFS Longitude"]].values)


tree = BallTree(subway_coords, metric="haversine")


distances, indices = tree.query(airbnb_coords, k=1)


earth_radius_km = 6371
ab_final["nearest_station_distance_km"] = distances[:, 0] * earth_radius_km


nearest_stations = mta_final.iloc[indices[:, 0]].reset_index(drop=True)


ab_final["nearest_station"] = nearest_stations["Stop Name"].values
ab_final["nearest_station_borough"] = nearest_stations["Borough"].values
ab_final["nearest_station_ada"] = nearest_stations["ADA"].values
ab_final["nearest_station_route_count"] = nearest_stations["route_count"].values
ab_final["nearest_station_complex_id"] = nearest_stations["Complex ID"].values


print(ab_final[[
    "name",
    "latitude",
    "longitude",
    "nearest_station",
    "nearest_station_distance_km",
    "nearest_station_borough",
    "nearest_station_ada",
    "nearest_station_route_count"
]].head())

                                               name  latitude  longitude  \
0                Clean & quiet apt home by the park  40.64749  -73.97237   
1                             Skylit Midtown Castle  40.75362  -73.98377   
2               THE VILLAGE OF HARLEM....NEW YORK !  40.80902  -73.94190   
3                   Cozy Entire Floor of Brownstone  40.68514  -73.95976   
4  Entire Apt: Spacious Studio/Loft by central park  40.79851  -73.94399   

      nearest_station  nearest_station_distance_km nearest_station_borough  \
0  Fort Hamilton Pkwy                     0.465364                      Bk   
1     42 St-Bryant Pk                     0.094923                       M   
2              125 St                     0.333712                       M   
3          Classon Av                     0.415913                      Bk   
4              116 St                     0.200187                       M   

   nearest_station_ada  nearest_station_route_count  
0                   

In [13]:
ab_final["distance_category"] = pd.cut(
    ab_final["nearest_station_distance_km"],
    bins=[0, 0.25, 0.5, 1, 2, float("inf")],
    labels=["0-250m", "250-500m", "500-1000m", "1-2km", ">2km"],
    include_lowest=True
)
ab_final.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,...,calculated_host_listings_count,availability_365,price_level_in_hood,nearest_station_distance_km,nearest_station,nearest_station_borough,nearest_station_ada,nearest_station_route_count,nearest_station_complex_id,distance_category
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,...,6,365,Medium,0.465364,Fort Hamilton Pkwy,Bk,0,2,242,250-500m
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,...,2,355,High,0.094923,42 St-Bryant Pk,M,0,4,609,0-250m
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,...,1,365,Medium,0.333712,125 St,M,0,2,439,250-500m
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,...,1,194,Medium,0.415913,Classon Av,Bk,0,1,290,250-500m
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,...,1,0,Low,0.200187,116 St,M,0,1,393,0-250m


In [14]:
print(ab_final.shape)
print(ab_final[[
    "nearest_station",
    "nearest_station_distance_km",
    "nearest_station_borough",
    "nearest_station_ada",
    "nearest_station_route_count",
    "distance_category"
]].head())

print(ab_final["nearest_station_distance_km"].describe())


(47125, 24)
      nearest_station  nearest_station_distance_km nearest_station_borough  \
0  Fort Hamilton Pkwy                     0.465364                      Bk   
1     42 St-Bryant Pk                     0.094923                       M   
2              125 St                     0.333712                       M   
3          Classon Av                     0.415913                      Bk   
4              116 St                     0.200187                       M   

   nearest_station_ada  nearest_station_route_count distance_category  
0                    0                            2          250-500m  
1                    0                            4            0-250m  
2                    0                            2          250-500m  
3                    0                            1          250-500m  
4                    0                            1            0-250m  
count    47125.000000
mean         0.433353
std          0.568042
min          0.000906

In [15]:
print(ab_final["distance_category"].value_counts(dropna=False))

distance_category
250-500m     20110
0-250m       16734
500-1000m     8057
1-2km         1286
>2km           938
Name: count, dtype: int64


In [16]:
ab_final.to_csv("NYC_Airbnb_Subway_Final.csv", index=False)

In [17]:
check_df = pd.read_csv("NYC_Airbnb_Subway_Final.csv")
print(check_df.shape)
print(check_df.head())

(47125, 24)
     id                                              name  host_id  \
0  2539                Clean & quiet apt home by the park     2787   
1  2595                             Skylit Midtown Castle     2845   
2  3647               THE VILLAGE OF HARLEM....NEW YORK !     4632   
3  3831                   Cozy Entire Floor of Brownstone     4869   
4  5022  Entire Apt: Spacious Studio/Loft by central park     7192   

     host_name neighbourhood_group neighbourhood  latitude  longitude  \
0         John            Brooklyn    Kensington  40.64749  -73.97237   
1     Jennifer           Manhattan       Midtown  40.75362  -73.98377   
2    Elisabeth           Manhattan        Harlem  40.80902  -73.94190   
3  LisaRoxanne            Brooklyn  Clinton Hill  40.68514  -73.95976   
4        Laura           Manhattan   East Harlem  40.79851  -73.94399   

         room_type  price  ...  calculated_host_listings_count  \
0     Private room    149  ...                               6

In [18]:
check_df.columns

Index(['id', 'name', 'host_id', 'host_name', 'neighbourhood_group',
       'neighbourhood', 'latitude', 'longitude', 'room_type', 'price',
       'minimum_nights', 'number_of_reviews', 'last_review',
       'reviews_per_month', 'calculated_host_listings_count',
       'availability_365', 'price_level_in_hood',
       'nearest_station_distance_km', 'nearest_station',
       'nearest_station_borough', 'nearest_station_ada',
       'nearest_station_route_count', 'nearest_station_complex_id',
       'distance_category'],
      dtype='object')